In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_4.py ──
"""
Shared infrastructure for MLFP04 Exercise 4 — Anomaly Detection and Ensembles.

Contains: data loading (e-commerce customers + rare-return anomaly label),
feature engineering, score normalisation helpers, metric reporting,
visualisation shortcuts.

Technique-specific code (Z-score thresholding, Isolation Forest fit, LOF
neighbour count, EnsembleEngine blend weights) does NOT belong here — it
lives in the per-technique files in `modules/mlfp04/solutions/ex_4/`.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
np.random.seed(42)

OUTPUT_DIR = Path("outputs") / "ex4_anomaly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANOMALY_QUANTILE = 0.99
FEATURE_BLOCKLIST = {
    "is_fraud",
    "customer_id",
    "ltv_tier",
    "product_categories",
    "review_text",
    "region",
    "device_type",
    "payment_method",
    "loyalty_member",
    "churned",
}


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customer data with rare-return anomaly label
# ════════════════════════════════════════════════════════════════════════
# The dataset ships with the mlfp03 module. We reuse it here because the
# anomaly story lives in the top 1% of return rates — a natural rare-event
# signal for unsupervised anomaly detection methods to find.


def load_anomaly_frame(quantile: float = ANOMALY_QUANTILE) -> pl.DataFrame:
    """Load e-commerce customers and attach a 1% rare-return anomaly label.

    Returns a polars DataFrame with an additional `is_fraud` column
    (1 where num_returns is in the top (1-quantile) percentile, else 0).
    """
    loader = MLFPDataLoader()
    raw = loader.load("mlfp03", "ecommerce_customers.parquet")
    threshold = raw["num_returns"].quantile(quantile)
    return raw.with_columns(
        (pl.col("num_returns") >= threshold).cast(pl.Int64).alias("is_fraud")
    )


def build_features(frame: pl.DataFrame) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Drop nulls, pick numeric features, standardise and return (X, y, cols).

    Returns standardised X (float64), y (int), and the feature column names.
    The returned X is suitable for sklearn-style anomaly detectors.
    """
    feature_cols = [c for c in frame.columns if c not in FEATURE_BLOCKLIST]
    X, y, _col_info = to_sklearn_input(
        frame.drop_nulls(),
        feature_columns=feature_cols,
        target_column="is_fraud",
    )
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X).astype(np.float64)
    return X_scaled, y.astype(int), feature_cols


def load_dataset() -> tuple[np.ndarray, np.ndarray, list[str], pl.DataFrame]:
    """One-call helper: load the frame, build features, return everything."""
    frame = load_anomaly_frame()
    X, y, cols = build_features(frame)
    return X, y, cols, frame


# ════════════════════════════════════════════════════════════════════════
# SCORE HELPERS
# ════════════════════════════════════════════════════════════════════════
# Anomaly detectors emit scores on wildly different scales. Normalising to
# [0, 1] (or to a rank) is what makes blending across methods possible.


def normalise_scores(scores: np.ndarray) -> np.ndarray:
    """Min-max normalise an anomaly score array to [0, 1]."""
    scores = np.asarray(scores, dtype=np.float64)
    span = scores.max() - scores.min()
    return (scores - scores.min()) / (span + 1e-10)


def rank_normalise(scores: np.ndarray) -> np.ndarray:
    """Convert an anomaly score array to percentile ranks in [0, 1]."""
    from scipy.stats import rankdata

    return rankdata(np.asarray(scores, dtype=np.float64)) / len(scores)


def score_metrics(y_true: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    """Return AUC-ROC and average precision (AUC-PR) for an anomaly score."""
    return {
        "auc_roc": float(roc_auc_score(y_true, scores)),
        "avg_precision": float(average_precision_score(y_true, scores)),
    }


def print_metrics(
    name: str, y_true: np.ndarray, scores: np.ndarray
) -> dict[str, float]:
    """Compute metrics, print them on one line, and return the dict."""
    m = score_metrics(y_true, scores)
    print(f"  {name:<24} AUC-ROC={m['auc_roc']:.4f}  " f"AP={m['avg_precision']:.4f}")
    return m


def precision_at_recall(
    y_true: np.ndarray, scores: np.ndarray, target_recall: float
) -> tuple[float, float]:
    """Return (precision, threshold) at the TIGHTEST point where recall >= target.

    sklearn returns precisions/recalls ordered by ascending threshold, so
    recall decreases as threshold increases. We want the highest threshold
    that still meets the recall target — i.e. the last index where recall
    is still >= the target, which gives the maximum precision for that
    recall level.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, scores)
    # Drop the sentinel last point (precision=1.0, recall=0.0, no threshold)
    ps = precisions[:-1]
    rs = recalls[:-1]
    ts = thresholds
    mask = rs >= target_recall
    if not mask.any():
        return float(ps[0]), float(ts[0])
    # The tightest threshold satisfying the recall target is the largest
    # index where mask is True (thresholds are ascending).
    idx = int(np.where(mask)[0][-1])
    return float(ps[idx]), float(ts[idx])


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION
# ════════════════════════════════════════════════════════════════════════


def write_comparison_chart(
    comparison: dict[str, dict[str, float]], filename: str
) -> Path:
    """Render a kailash-ml ModelVisualizer metric_comparison chart to HTML."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.metric_comparison(comparison)
    fig.update_layout(title="Anomaly Detection Method Comparison")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def write_roc_chart(
    y_true: np.ndarray, scores: np.ndarray, name: str, filename: str
) -> Path:
    """Render a ROC curve for a single detector."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.roc_curve(y_true, scores)
    fig.update_layout(title=f"ROC — {name}")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def write_monitoring_chart(anomaly_rates: list[float], filename: str) -> Path:
    """Render an anomaly-rate-over-time chart for production monitoring."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.training_history(
        {"Anomaly Rate %": [r * 100 for r in anomaly_rates]},
        x_label="Time Window",
    )
    fig.update_layout(title="Anomaly Rate Over Time (Production Monitoring)")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


# ════════════════════════════════════════════════════════════════════════
# ENSEMBLE ADAPTER
# ════════════════════════════════════════════════════════════════════════
# kailash-ml EnsembleEngine.blend() expects estimator-shaped objects with
# predict_proba. Each detector in this exercise has already produced a
# score vector, so we wrap those vectors in a minimal estimator.


class AnomalyScoreEstimator:
    """Minimal sklearn-shaped wrapper exposing precomputed scores.

    EnsembleEngine.blend() calls predict_proba(X) on every estimator and
    averages the resulting class-1 probabilities. This adapter normalises
    the underlying scores to [0, 1] and returns them as P(anomaly).
    """

    def __init__(self, scores: np.ndarray):
        self._scores = np.asarray(scores, dtype=np.float64)
        self._norm = normalise_scores(self._scores)
        self.classes_ = np.array([0, 1])

    def fit(self, X: Any, y: Any = None) -> "AnomalyScoreEstimator":
        return self

    def predict_proba(self, X: Any) -> np.ndarray:
        n = len(X) if hasattr(X, "__len__") else self._norm.shape[0]
        norm = self._norm[:n]
        return np.column_stack([1.0 - norm, norm])

    def predict(self, X: Any) -> np.ndarray:
        n = len(X) if hasattr(X, "__len__") else self._scores.shape[0]
        threshold = float(np.median(self._scores))
        return (self._scores[:n] > threshold).astype(int)




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 4.3: Local Outlier Factor (LOF)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Explain LOF as a ratio of neighbour density to point density
#   - Sweep n_neighbors and explain the "locality" trade-off
#   - Fit LOF and turn negative_outlier_factor_ into an anomaly score
#   - Explain when LOF beats Isolation Forest (varying-density clusters)
#
# PREREQUISITES: 4.2 (Isolation Forest).
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — comparing LOCAL densities catches embedded anomalies
#   2. Build — sweep n_neighbors and pick the best value
#   3. Train — fit LOF and extract negative_outlier_factor_
#   4. Visualise — ROC curve (written to outputs/)
#   5. Apply — Shopee return-fraud cluster detection
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.neighbors import LocalOutlierFactor



## THEORY — Local Density Beats Global Distance


In [ ]:
# LOF(p) = mean( density(neighbour_i) / density(p) ).
# LOF ~ 1 means p has roughly the same density as its k-nearest neighbours.
# LOF >> 1 means p sits in a sparser pocket than its neighbours = anomaly.
# LOF catches cluster-embedded anomalies that Isolation Forest misses.



## TASK 2 — BUILD: sweep n_neighbors


In [ ]:
X, y, _feature_cols, _frame = load_dataset()
n_samples, n_features = X.shape
print("\n" + "=" * 70)
print("  Local Outlier Factor (LOF)")
print("=" * 70)
print(
    f"Rows: {n_samples:,} | Features: {n_features} | "
    f"Anomalies: {int(y.sum()):,} ({y.mean():.2%})"
)

print("\nn_neighbors sweep:")
for n_nbrs in [10, 20, 30, 50]:
    # TODO: Build LocalOutlierFactor with n_neighbors=n_nbrs,
    # contamination=0.01, novelty=False
    lof_test = ____

    # TODO: Run fit_predict(X) to get labels
    labels_test = ____

    # TODO: Turn negative_outlier_factor_ into an anomaly score where
    # HIGHER means more anomalous. Hint: negate the attribute.
    scores_test = ____

    m = score_metrics(y, scores_test)
    n_flagged = int((labels_test == -1).sum())
    print(
        f"  n_neighbors={n_nbrs:<3}  AUC-ROC={m['auc_roc']:.4f}  "
        f"AP={m['avg_precision']:.4f}  flagged={n_flagged:,}"
    )



## TASK 3 — TRAIN: fit LOF with n_neighbors=20


In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01, novelty=False)
lof_labels = lof.fit_predict(X)
lof_scores = -lof.negative_outlier_factor_

print("\nFinal LOF (n_neighbors=20):")
lof_metrics = print_metrics("LOF", y, lof_scores)
print(f"  Predicted anomalies: {int((lof_labels == -1).sum()):,}")
print(f"  True anomalies:      {int(y.sum()):,}")



### Checkpoint


In [ ]:
assert (
    lof_metrics["auc_roc"] > 0.5
), f"LOF AUC-ROC {lof_metrics['auc_roc']:.4f} should beat random"
assert lof_scores.std() > 0, "LOF scores should vary across rows"
assert lof_scores.shape[0] == n_samples, "Score length must match row count"
print("\n[ok] Checkpoint passed — LOF scored all rows\n")



## TASK 4 — VISUALISE: ROC curve


In [ ]:
roc_path = write_roc_chart(y, lof_scores, "LOF", "ex4_roc_lof.html")
print(f"Saved ROC chart: {roc_path}")

print("\nLOF catches cluster-embedded anomalies.")
print("Isolation Forest catches globally-distant anomalies.")
print("Use BOTH, then blend — see 04_ensemble_blending.py.")



## TASK 5 — APPLY: Shopee Return-Fraud Cluster Detection


In [ ]:
# Shopee's refund-ring fraud forms a tight cluster of accounts embedded
# in the buyer population. Isolation Forest misses it; LOF catches it
# because the fraud cluster has high LOCAL density relative to its
# neighbourhood of legitimate buyers.
#
# Impact: ~S$5.6M/year recovered against ~S$40K/year infrastructure cost.



## REFLECTION


[x] LOF as a density-ratio test
  [x] n_neighbors as the "locality" knob
  [x] Cluster-embedded anomaly detection
  [x] O(n^2) scalability limit awareness
  [x] Shopee refund-ring detection scenario

  Next: 04_ensemble_blending.py — combine all four detectors with
  kailash-ml EnsembleEngine.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

